In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :memoryless

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [memoryless_model] Fitting chain 1 (tau=34)
[ Info: [memoryless] iter 1000/1000000 elapsed=3.6s, rate=0.221, mean=[1.132, 0.00128, 1.654], std=[0.1387, 0.000538, 0.1864] [ADAPT]
[ Info: [memoryless] iter 2000/1000000 elapsed=6.8s, rate=0.210, mean=[1.144, 0.00134, 2.054], std=[0.1005, 0.000467, 0.3972] [ADAPT]
[ Info: [memoryless] iter 3000/1000000 elapsed=8.9s, rate=0.215, mean=[1.159, 0.00133, 2.280], std=[0.0859, 0.000402, 0.4379] [ADAPT]
[ Info: [memoryless] iter 4000/1000000 elapsed=11.0s, rate=0.216, mean=[1.175, 0.00131, 2.430], std=[0.0801, 0.000362, 0.4523] [ADAPT]
[ Info: [memoryless] iter 5000/1000000 elapsed=13.2s, rate=0.217, mean=[1.173, 0.00133, 2.459], std=[0.0728, 0.000340, 0.4127] [ADAPT]
[ Info: [memoryless] iter 6000/1000000 elapsed=15.3s, rate=0.218, mean=[1.174, 0.00135, 2.493], std=[0.0681, 0.000349, 0.3852] [ADAPT]
[ Info: [memoryless] iter 7000/1000000 elapsed=17.5s, rate=0.218, mean=[1.175, 0.00135, 2.508], std=[0.0639, 0.000333, 0.3619] [ADAPT]
[ Info